<a href="https://colab.research.google.com/github/naceurarij2-design/projet_2/blob/main/Checkpoint_20_Fine_Tune_BERT_for_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q --upgrade transformers datasets evaluate accelerate scikit-learn

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:


from datasets import load_dataset

raw = load_dataset("imdb")

train_raw = raw["train"].shuffle(seed=42).select(range(4000))
test_raw  = raw["test"].shuffle(seed=42).select(range(2000))


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)

tokenized_train = train_raw.map(tokenize_fn, batched=True)
tokenized_test  = test_raw.map(tokenize_fn, batched=True)


tokenized_train = tokenized_train.remove_columns([c for c in tokenized_train.column_names if c not in ("input_ids","attention_mask","label")])
tokenized_test  = tokenized_test.remove_columns([c for c in tokenized_test.column_names if c not in ("input_ids","attention_mask","label")])

tokenized_train = tokenized_train.rename_column("label", "labels").with_format("torch")
tokenized_test  = tokenized_test.rename_column("label", "labels").with_format("torch")


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=500,   # log every 500 steps
    save_steps=500,      # save checkpoint every 500 steps
    report_to="none",
)


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from transformers import Trainer
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"]
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,   # already tokenized
    eval_dataset=tokenized_test,     # already tokenized
    compute_metrics=compute_metrics, # metrics still work
)


In [ ]:
trainer.train()
trainer.save_model("./final_model")

metrics = trainer.evaluate()
print(metrics)


Step,Training Loss
500,0.388717
1000,0.225443
1500,0.095366


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step,Accuracy,F1
0.095366,0.461096,1500,0.900500,0.902499


{'eval_loss': 0.4610959589481354, 'eval_accuracy': 0.9005, 'eval_f1': 0.9024987751102401}


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import torch
from torch.utils.data import DataLoader
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

loader = DataLoader(tokenized_test, batch_size=8)
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in loader:
        labels = batch["labels"].to(device)
        inputs = {k:v.to(device) for k,v in batch.items() if k!="labels"}
        outputs = model(**inputs)
        preds = outputs.logits.argmax(dim=-1).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(labels.cpu().numpy())

preds = np.concatenate(all_preds)
labels = np.concatenate(all_labels)

print(classification_report(labels, preds, digits=4))
print("Confusion matrix:\n", confusion_matrix(labels, preds))


              precision    recall  f1-score   support

           0     0.9176    0.8800    0.8984      1000
           1     0.8847    0.9210    0.9025      1000

    accuracy                         0.9005      2000
   macro avg     0.9012    0.9005    0.9005      2000
weighted avg     0.9012    0.9005    0.9005      2000

Confusion matrix:
 [[880 120]
 [ 79 921]]


In [ ]:
# test with my own input
from transformers import pipeline

sentiment = pipeline("sentiment-analysis", model="./final_model", tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

print(sentiment(["I loved this movie!", "This film was awful."]))
# BERT performed quite well onthe  sentiment classification task: with ~90% accuracy,
#both classes are balanced in precision and recall, and the F1‑scores are strong.
#To improve performance further, I could try DistilBERT for faster runs, experiment with hyperparameter tuning,
#or apply data augmentation to enrich the training set.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'LABEL_1', 'score': 0.998941957950592}, {'label': 'LABEL_0', 'score': 0.9988196492195129}]
